In [ ]:
import pandas as pd
import re

# ----------------------------
# LOAD RESULTS
# ----------------------------
df = pd.read_csv("/content/apc_results.csv")


# ----------------------------
# HELPER FUNCTION FOR MATH
# ----------------------------
def contains_answer(text, answer):
    if pd.isna(text):
        return False
    return str(answer).lower() in str(text).lower()


# ----------------------------
# HELPER FUNCTION FOR HALLUCINATION
# ----------------------------
def normalize_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def semantic_match(ground_truth, answer, threshold=0.4):
    gt = normalize_text(ground_truth)
    ans = normalize_text(answer)

    if not gt or not ans:
        return False

    stopwords = {
        "the", "a", "an", "is", "are", "was", "were", "to", "of", "in", "on", "at",
        "for", "and", "or", "that", "this", "it", "as", "with", "by", "be", "from",
        "into", "your", "you", "their", "there", "they", "them", "will", "would",
        "can", "could", "should", "has", "have", "had", "not", "no", "does", "do",
        "did", "than", "then", "about", "after", "before", "through", "very"
    }

    gt_words = [w for w in gt.split() if len(w) > 3 and w not in stopwords]

    if len(gt_words) == 0:
        return False

    hits = sum(1 for w in gt_words if w in ans)
    score = hits / len(gt_words)

    return score >= threshold


# ----------------------------
# MATH ACCURACY (UNCHANGED)
# ----------------------------
math_df = df[df["dataset"] == "math"]

baseline_correct = 0
static_correct = 0
apc_correct = 0

for _, row in math_df.iterrows():
    gt = str(row["ground_truth"])

    if contains_answer(row["baseline"], gt):
        baseline_correct += 1

    if contains_answer(row["static_chain"], gt):
        static_correct += 1

    if contains_answer(row["apc_final"], gt):
        apc_correct += 1

math_total = len(math_df)

math_results = {
    "Baseline Accuracy": baseline_correct / math_total if math_total else 0,
    "Static Chain Accuracy": static_correct / math_total if math_total else 0,
    "APC Accuracy": apc_correct / math_total if math_total else 0
}


# ----------------------------
# HALLUCINATION RATE (FIXED)
# ----------------------------
hall_df = df[df["dataset"] == "hallucination"]

hallucinations_baseline = 0
hallucinations_static = 0
hallucinations_apc = 0

for _, row in hall_df.iterrows():
    gt = row["ground_truth"]

    if not semantic_match(gt, row["baseline"]):
        hallucinations_baseline += 1

    if not semantic_match(gt, row["static_chain"]):
        hallucinations_static += 1

    if not semantic_match(gt, row["apc_final"]):
        hallucinations_apc += 1

hall_total = len(hall_df)

hall_results = {
    "Baseline Hallucination Rate": hallucinations_baseline / hall_total if hall_total else 0,
    "Static Chain Hallucination Rate": hallucinations_static / hall_total if hall_total else 0,
    "APC Hallucination Rate": hallucinations_apc / hall_total if hall_total else 0
}


# ----------------------------
# REFINEMENT RATE
# ----------------------------
refinements = df["refinement_triggered"].sum()
total_questions = len(df)

refinement_rate = refinements / total_questions if total_questions else 0


# ----------------------------
# PRINT RESULTS
# ----------------------------
print("\n----- MATH ACCURACY -----")
for k, v in math_results.items():
    print(f"{k}: {round(v * 100, 2)}%")

print("\n----- HALLUCINATION RATE -----")
for k, v in hall_results.items():
    print(f"{k}: {round(v * 100, 2)}%")

print("\n----- APC REFINEMENT RATE -----")
print(f"Refinements Triggered: {refinements}")
print(f"Refinement Rate: {round(refinement_rate * 100, 2)}%")


# ----------------------------
# SAVE TABLE FOR PAPER
# ----------------------------
summary = pd.DataFrame({
    "Metric": [
        "Baseline Math Accuracy",
        "Static Chain Math Accuracy",
        "APC Math Accuracy",
        "Baseline Hallucination Rate",
        "Static Chain Hallucination Rate",
        "APC Hallucination Rate",
        "APC Refinement Rate"
    ],
    "Value": [
        math_results["Baseline Accuracy"],
        math_results["Static Chain Accuracy"],
        math_results["APC Accuracy"],
        hall_results["Baseline Hallucination Rate"],
        hall_results["Static Chain Hallucination Rate"],
        hall_results["APC Hallucination Rate"],
        refinement_rate
    ]
})

summary.to_csv("experiment_summary.csv", index=False)

print("\nSummary table saved as experiment_summary.csv")


----- MATH ACCURACY -----
Baseline Accuracy: 90.0%
Static Chain Accuracy: 92.0%
APC Accuracy: 94.0%

----- HALLUCINATION RATE -----
Baseline Hallucination Rate: 12.0%
Static Chain Hallucination Rate: 8.0%
APC Hallucination Rate: 6.0%

----- APC REFINEMENT RATE -----
Refinements Triggered: 72
Refinement Rate: 48.0%

Summary table saved as experiment_summary.csv
